## Support Vector Machine (SVM)

O [SVM](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) é um modelo que busca encontrar o melhor hiperplano que separa as classes no espaço de características.

Ele pode utilizar diferentes funções de kernel para lidar com relações lineares e não lineares.

### Hiperparâmetros utilizados

- **C**: controla a margem de separação
  - Valores altos → menor margem (mais complexo)
  - Valores baixos → maior margem (mais simples)

- **kernel**: define o tipo de separação
  - `linear`: separação linear
  - `rbf`: separação não linear (default)

### Estratégia

Foi utilizado GridSearchCV com validação cruzada para identificar a melhor combinação de parâmetros, utilizando a métrica ROC AUC.

In [2]:
import pandas as pd
import sys
import os

# add raiz do projeto
sys.path.append(os.path.abspath(".."))

from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

from preprocessing.main_preprocessing import load_and_preprocess
from utils.experiment_logger import save_results


## BASELINE

In [2]:
scenarios = [
    "sem_submodalidade",
    "submodalidade_agrupada",
    "submodalidade_engineered"
]

smote_options = [False, True]

results = []

for scenario in scenarios:
    for use_smote in smote_options:

        print(f"\n🔎 Scenario: {scenario} | SMOTE: {use_smote}")

        X_train, X_test, y_train, y_test = load_and_preprocess(
            "../predictive_models/scrdata_202505.csv",
            scenario=scenario,
            use_smote=use_smote
        )

        steps = []
        steps.append(('scaler', StandardScaler()))
        if use_smote:
            steps.append(('smote', SMOTE(random_state=42)))
        steps.append(('svc', SVC(probability=True, class_weight="balanced", random_state=42)))

        model = Pipeline(steps)

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

        results.append({
            "model": "SVM",
            "scenario": scenario,
            "smote": use_smote,
            "roc_auc": roc_auc_score(y_test, y_proba),
            "f1": f1_score(y_test, y_pred),
            "accuracy": accuracy_score(y_test, y_pred),
            "n_features": X_train.shape[1],
            "phase": "baseline",
            "timestamp": pd.Timestamp.now()
        })


save_results(results, file_path="../results/experiments.csv")

display(pd.DataFrame(results))



🔎 Scenario: sem_submodalidade | SMOTE: False

🔎 Scenario: sem_submodalidade | SMOTE: True

🔎 Scenario: submodalidade_agrupada | SMOTE: False

🔎 Scenario: submodalidade_agrupada | SMOTE: True

🔎 Scenario: submodalidade_engineered | SMOTE: False

🔎 Scenario: submodalidade_engineered | SMOTE: True


,model,scenario,smote,roc_auc,f1,accuracy,n_features,phase,timestamp
0,SVM,sem_submodalidade,False,0.600821,0.314266,0.502026,67,baseline,2026-04-10 01:52:01.996902
1,SVM,sem_submodalidade,True,0.601294,0.321710,0.504745,67,baseline,2026-04-10 05:40:28.806795
2,SVM,submodalidade_agrupada,False,0.600820,0.314181,0.501989,97,baseline,2026-04-10 08:47:56.596771
3,SVM,submodalidade_agrupada,True,0.601302,0.321752,0.504764,97,baseline,2026-04-10 13:25:50.764645
4,SVM,submodalidade_engineered,False,0.600821,0.314266,0.502026,67,baseline,2026-04-10 15:59:55.428200
5,SVM,submodalidade_engineered,True,0.601294,0.321710,0.504745,67,baseline,2026-04-10 19:49:02.943785


## GRIDSEARCH

In [ ]:
X_train, X_test, y_train, y_test = load_and_preprocess(
    "../predictive_models/scrdata_202505.csv",
    scenario="sem_submodalidade",
    use_smote=False
)


param_grid_svm = {
    "smote": [SMOTE(random_state=42), "passthrough"],
    "svc__C": [0.1, 1, 10],
    "svc__kernel": ["linear"]
}


grid_svm = GridSearchCV(
    estimator=Pipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('svc', SVC(probability=True,
        class_weight="balanced",
        random_state=42))
    ]),
    param_grid=param_grid_svm,
    cv=5,                
    scoring="roc_auc",
    n_jobs=2
)

grid_svm.fit(X_train, y_train)

print("Best parameters:", grid_svm.best_params_)
print("Best ROC AUC (CV):", grid_svm.best_score_)


#? BEST MODEL TEST EVALUATION

best_svm = grid_svm.best_estimator_

y_pred = best_svm.predict(X_test)
y_proba = best_svm.predict_proba(X_test)[:, 1]


#? TUNING (CV)

tuning_results = []

cv_results = pd.DataFrame(grid_svm.cv_results_)
cv_results['smote_used'] = cv_results['param_smote'].apply(lambda x: x != 'passthrough')

for smote_val in [True, False]:
    sub_results = cv_results[cv_results['smote_used'] == smote_val]
    if not sub_results.empty:
        best_row = sub_results.sort_values('mean_test_score', ascending=False).iloc[0]
        params = best_row['params']
        
        tuning_results.append({
            "model": "SVM",
            "scenario": "sem_submodalidade",
            "smote": smote_val,
            "phase": "tuning_cv",
            "roc_auc": best_row['mean_test_score'],
            "f1": None,
            "accuracy": None,
            "best_params": str(params),
            "timestamp": pd.Timestamp.now()
        })
        
        # Re-fit the best model for this group to get test metrics
        best_model_group = grid_svm.estimator.set_params(**params)
        best_model_group.fit(X_train, y_train)
        
        y_pred = best_model_group.predict(X_test)
        y_proba = best_model_group.predict_proba(X_test)[:, 1]
        
        tuning_results.append({
            "model": "SVM",
            "scenario": "sem_submodalidade",
            "smote": smote_val,
            "phase": "test",
            "roc_auc": roc_auc_score(y_test, y_proba),
            "f1": f1_score(y_test, y_pred),
            "accuracy": accuracy_score(y_test, y_pred),
            "best_params": str(params),
            "timestamp": pd.Timestamp.now()
        })

save_results(tuning_results, file_path="../results/model_results.csv")

display(pd.DataFrame(tuning_results))

KeyboardInterrupt: 